# Acno: training our three skin models

**Team:** Travis, Matthew, Alan, Tanner, Erwin (Inspirit AI)

**Motto:** Scan smarter. Skin clearer.

Acno is an app that helps teens understand their skin. You scan your face and the app
tells you your skin type, what kind of acne it sees, how serious it looks, and gives
you a simple routine to take care of it.

This notebook is where we train the three models that power the app:

1. **Skin type classifier**: is the skin dry, normal, or oily?
2. **Acne type classifier**: whiteheads, blackheads, papules, pustules, or cysts?
3. **Lesion detector**: finds every acne spot on the face. Counting the spots gives us
   a severity score (clear, mild, moderate, or severe).

We train on three public Kaggle datasets. Full details, licenses, and known risks are
written up in [docs/DATASETS.md](https://github.com/matthewyongenwang-coder/acno/blob/main/docs/DATASETS.md)
in our repo. Two important notes:

- These datasets contain photos of real people, so we never copy them into our GitHub
  repo. This notebook downloads them fresh each time.
- Everything the models say is educational guidance, not a medical diagnosis. The app
  always tells users with severe findings to see a dermatologist.

**How to run this:** Runtime > Change runtime type > pick a GPU (T4 is fine), then
Runtime > Run all. Training everything takes roughly 30 to 45 minutes for T4.


## 1. Setup

Colab already has TensorFlow. We add `kagglehub` (dataset downloads, no account needed)
and `ultralytics` (the YOLO library for our lesion detector).

`QUICK_TEST` is a switch for debugging: when it is `True`, everything trains on a tiny
slice of data for a couple of epochs so we can check the whole notebook runs without
waiting for real training. Leave it `False` for the real runs.

In [ ]:
%pip install -q kagglehub ultralytics

QUICK_TEST = False  # set True to smoke-test the notebook in a few minutes

In [ ]:
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus[0].name if gpus else "none found (training will be slow, switch the runtime to a GPU)")

SEED = 77
tf.random.set_seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path("data/raw")
RESULTS = Path("results")
MODELS = Path("models")
RESULTS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

## 2. Get the data

Same download logic as `scripts/download_data.py` in our repo: kagglehub pulls each
dataset into its cache, then we copy it into `data/raw/` so all paths below are
predictable.

In [ ]:
import kagglehub

# name -> (kaggle slug, subfolder inside the download that holds the data)
DATASETS = {
    "skin_type": ("shakyadissanayake/oily-dry-and-normal-skin-types-dataset", "Oily-Dry-Skin-Types"),
    "acne_type": ("tiswan14/acne-dataset-image", "AcneDataset"),
    "acne_yolo": ("osmankagankurnaz/acne-dataset-in-yolov8-format", "data-2"),
}

for name, (slug, subdir) in DATASETS.items():
    dest = DATA_DIR / name
    if dest.exists():
        print(f"[skip] {name}: already downloaded")
        continue
    print(f"[download] {name} <- {slug}")
    cache_path = Path(kagglehub.dataset_download(slug))
    src = cache_path / subdir
    if not src.exists():
        src = cache_path  # layout changed upstream, take the whole thing
    shutil.copytree(src, dest)
    print(f"  -> {dest}")

print("done")

In [ ]:
len(DATASETS)

### What the data looks like

Before training anything, we count how many images each class has. This matters because
our models can cheat: if 40% of the images are "normal" skin, a lazy model that always
answers "normal" is right 40% of the time. Knowing the imbalance up front tells us
where to correct for it (we use class weights later).

In [ ]:
def count_images(split_dir):
    counts = {}
    for class_dir in sorted(split_dir.iterdir()):
        if class_dir.is_dir():
            counts[class_dir.name] = sum(1 for f in class_dir.rglob("*") if f.is_file())
    return counts

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, name in zip(axes, ["skin_type", "acne_type"]):
    counts = count_images(DATA_DIR / name / "train")
    ax.bar(counts.keys(), counts.values(), color="#7fb3a4")
    ax.set_title(f"{name}: training images per class")
    ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(RESULTS / "class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

for name in ["skin_type", "acne_type"]:
    print(name, count_images(DATA_DIR / name / "train"))

Takeaways: in `skin_type` the dry class is clearly under-represented, and in
`acne_type` the Whiteheads class has about 4 times fewer images than the others. We
will hand both models class weights so mistakes on rare classes cost more.

Here are a few sample images so we know what the models are actually looking at:

In [ ]:
def show_samples(dataset_name, per_class=3):
    train_dir = DATA_DIR / dataset_name / "train"
    class_dirs = sorted(d for d in train_dir.iterdir() if d.is_dir())
    fig, axes = plt.subplots(len(class_dirs), per_class, figsize=(per_class * 2.4, len(class_dirs) * 2.4))
    for row, class_dir in zip(axes, class_dirs):
        images = sorted(f for f in class_dir.rglob("*") if f.is_file())[:per_class]
        for ax, img_path in zip(row, images):
            ax.imshow(plt.imread(img_path))
            ax.set_title(class_dir.name, fontsize=9)
            ax.axis("off")
    fig.tight_layout()
    plt.show()

show_samples("skin_type")

## 3. Shared training recipe

Both classifiers use the same recipe, so we write it once as a few helper functions:

- **Transfer learning.** Our datasets are small (a few thousand images). Training a
  network from scratch on that would overfit badly. Instead we start from MobileNetV2,
  a network already trained on 1.4 million everyday photos, and only teach it our
  specific skin classes. It already knows about edges, textures, and redness; we just
  add the last step.
- **Augmentation.** We randomly flip, rotate, and brighten training images a little.
  This makes the model less picky about lighting and camera angle, which is exactly
  the messy reality of teens taking selfies.
- **Class weights.** Rare classes count for more in the loss, so the model cannot
  ignore them.
- **Two training stages.** First we train only our new top layer (the backbone stays
  frozen). Then we unfreeze the last chunk of the backbone and fine-tune everything
  with a tiny learning rate.

In [ ]:
IMG_SIZE = (224, 224)
BATCH = 32

def load_split(root, split, shuffle):
    ds = tf.keras.utils.image_dataset_from_directory(
        root / split,
        image_size=IMG_SIZE,
        batch_size=BATCH,
        shuffle=shuffle,
        seed=SEED,
    )
    class_names = ds.class_names
    if QUICK_TEST:
        ds = ds.take(4)
    return ds.prefetch(tf.data.AUTOTUNE), class_names

def make_class_weights(train_dir, class_names):
    counts = count_images(train_dir)
    total = sum(counts.values())
    n = len(class_names)
    weights = {i: total / (n * counts[name]) for i, name in enumerate(class_names)}
    print("class weights:", {class_names[i]: round(w, 2) for i, w in weights.items()})
    return weights

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.06),
    tf.keras.layers.RandomBrightness(0.15),
    tf.keras.layers.RandomContrast(0.15),
], name="augment")

def build_classifier(num_classes):
    base = tf.keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
    base.trainable = False
    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = augment(inputs)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.25)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs), base

def train_classifier(name, root):
    train_ds, class_names = load_split(root, "train", shuffle=True)
    val_ds, _ = load_split(root, "valid", shuffle=False)
    weights = make_class_weights(root / "train", class_names)
    model, base = build_classifier(len(class_names))

    # stage 1: only the new top layer learns
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    epochs1 = 2 if QUICK_TEST else 8
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=epochs1, class_weight=weights)

    # stage 2: unfreeze the top of the backbone and fine-tune gently
    base.trainable = True
    for layer in base.layers[:-40]:
        layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    epochs2 = 1 if QUICK_TEST else 6
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=epochs2, class_weight=weights)

    history = {k: h1.history[k] + h2.history[k] for k in h1.history}
    model.save(MODELS / f"{name}.keras")
    return model, class_names, history

def plot_history(name, history):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, metric in zip(axes, ["accuracy", "loss"]):
        ax.plot(history[metric], label="train")
        ax.plot(history["val_" + metric], label="validation")
        ax.set_title(f"{name}: {metric}")
        ax.set_xlabel("epoch")
        ax.legend()
    fig.tight_layout()
    fig.savefig(RESULTS / f"{name}_training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()

def evaluate_classifier(name, model, root, class_names):
    test_ds, _ = load_split(root, "test", shuffle=False)
    y_true = np.concatenate([y.numpy() for _, y in test_ds])
    y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)
    labels = list(range(len(class_names)))
    print(classification_report(y_true, y_pred, labels=labels,
                                target_names=class_names, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    disp.plot(ax=ax, cmap="Greens", colorbar=False)
    ax.set_title(f"{name}: test set confusion matrix")
    plt.xticks(rotation=25)
    fig.tight_layout()
    fig.savefig(RESULTS / f"{name}_confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()
    return y_true, y_pred

## 4. Model 1: skin type (dry / normal / oily)

This model answers the first question a user has: what kind of skin do I have? That
answer drives which routine the app recommends (dry skin needs gentle hydration, oily
skin needs oil control, and so on).

One honest limitation we state everywhere: this dataset only covers dry, normal, and
oily. Combination and sensitive skin are not in the training data, so the app never
claims to detect them.

In [ ]:
skin_model, skin_classes, skin_history = train_classifier("skin_type", DATA_DIR / "skin_type")

In [ ]:
plot_history("skin_type", skin_history)
_ = evaluate_classifier("skin_type", skin_model, DATA_DIR / "skin_type", skin_classes)

Reading the confusion matrix: rows are what the skin was, columns are what
the model guessed. A perfect model only has numbers on the diagonal. Watch the dry row
especially, since dry had the fewest training images.

## 5. Model 2: acne type (whiteheads / blackheads / papules / pustules / cysts)

Different acne types need different care. Blackheads and whiteheads are clogged pores
and respond to gentle exfoliating ingredients. Papules and pustules are inflamed and
need calmer treatment. Cysts are the serious ones: deep, painful, and the app's answer
for them is always "please see a dermatologist".

Same recipe as model 1. The one thing to watch is Whiteheads, which has about 4 times
fewer images than the other classes. The class weights printed below show how much
harder the model gets punished for missing them.

In [ ]:
acne_model, acne_classes, acne_history = train_classifier("acne_type", DATA_DIR / "acne_type")

In [ ]:
plot_history("acne_type", acne_history)
_ = evaluate_classifier("acne_type", acne_model, DATA_DIR / "acne_type", acne_classes)

## 6. Model 3: lesion detector (how severe is it?)

The two classifiers say what kind of skin and acne they see. This third model answers
"how much": it draws a box around every individual acne spot it finds.

We use YOLOv8-nano, a small fast object detector, trained on faces where every lesion
was labeled by the dataset authors (derived from the ACNE04 research dataset). Counting
the boxes gives us a severity score. This is the same idea as the Hayashi grading scale
that dermatologists use, where severity is judged from lesion counts:

| lesions found | severity |
|---|---|
| 0 to 1 | clear |
| 2 to 10 | mild |
| 11 to 25 | moderate |
| more than 25 | severe |

A count is also easy to explain to users, and easy to track over time ("12 last week,
8 today").

In [ ]:
from ultralytics import YOLO

yolo = YOLO("yolov8n.pt")
yolo_results = yolo.train(
    data=str((DATA_DIR / "acne_yolo" / "data.yaml").resolve()),
    epochs=2 if QUICK_TEST else 40,
    imgsz=320 if QUICK_TEST else 640,
    fraction=0.1 if QUICK_TEST else 1.0,
    batch=16,
    seed=SEED,
    project=str(RESULTS),
    name="acne_yolo",
    exist_ok=True,
)
# ask the trainer where it saved the weights instead of guessing the path
best_weights = Path(yolo.trainer.best)
if not best_weights.exists():
    best_weights = Path(yolo.trainer.last)
shutil.copy(best_weights, MODELS / "acne_yolo.pt")
print("saved", MODELS / "acne_yolo.pt")

Ultralytics saves its own charts (losses, precision, recall, mAP) into
the run folder it prints above. The number to quote on the results slide is mAP50: how often the
boxes the model draws match the labeled lesions. Below we run the trained detector on a
few unseen test faces and count lesions to get severities.

In [ ]:
def severity_from_count(count):
    if count <= 1:
        return "clear"
    if count <= 10:
        return "mild"
    if count <= 25:
        return "moderate"
    return "severe"

detector = YOLO(str(MODELS / "acne_yolo.pt"))
test_images = sorted((DATA_DIR / "acne_yolo" / "test" / "images").glob("*"))[:6]
preds = detector.predict([str(p) for p in test_images], conf=0.25, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, pred in zip(axes.flat, preds):
    count = len(pred.boxes)
    ax.imshow(pred.plot()[:, :, ::-1])  # plot() returns BGR, flip to RGB
    ax.set_title(f"{count} lesions -> {severity_from_count(count)}")
    ax.axis("off")
fig.tight_layout()
fig.savefig(RESULTS / "yolo_sample_detections.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Why did the model say that? (Grad-CAM)

A skin app that just outputs "oily, moderate acne" with no reason is hard to trust.
Grad-CAM lets us peek inside a classifier: it highlights which parts of the photo
pushed the model toward its answer. If the skin type model is looking at the shiny
T-zone, good. If it is looking at the background, we have a problem.

We use these heatmaps in the app and on the explainability slide.

In [ ]:
def grad_cam(model, image_path, last_conv="Conv_1"):
    img = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    array = tf.keras.utils.img_to_array(img)[None, ...]

    # rebuild the forward pass so we can watch the last convolution layer
    base = next(l for l in model.layers if l.name.startswith("mobilenetv2"))
    conv_layer = base.get_layer(last_conv)
    cam_model = tf.keras.Model(base.input, [conv_layer.output, base.output])

    x = tf.keras.applications.mobilenet_v2.preprocess_input(array.copy())
    with tf.GradientTape() as tape:
        conv_out, base_out = cam_model(x)
        pooled = tf.reduce_mean(base_out, axis=[1, 2])
        # apply the model's own head to get class scores
        head_in = pooled
        for layer in model.layers:
            if isinstance(layer, tf.keras.layers.Dense):
                head_in = layer(head_in)
        top_class = tf.argmax(head_in[0])
        score = head_in[:, top_class]
    grads = tape.gradient(score, conv_out)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = tf.reduce_sum(conv_out[0] * weights, axis=-1)
    cam = tf.nn.relu(cam) / (tf.reduce_max(cam) + 1e-8)
    return img, cam.numpy(), int(top_class)

sample = sorted((DATA_DIR / "skin_type" / "test" / "oily").rglob("*"))[0]
img, cam, top = grad_cam(skin_model, sample)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(img); axes[0].set_title("input"); axes[0].axis("off")
axes[1].imshow(img); axes[1].imshow(cam, cmap="jet", alpha=0.45, extent=(0, IMG_SIZE[0], IMG_SIZE[1], 0))
axes[1].set_title(f"model looked here (predicted: {skin_classes[top]})"); axes[1].axis("off")
fig.tight_layout()
fig.savefig(RESULTS / "grad_cam_skin_type.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Export everything

Two things leave this notebook:

- `models/`: the three trained model files the app loads
  (`skin_type.keras`, `acne_type.keras`, `acne_yolo.pt`)
- `results/`: every chart above, ready for the results slides

Run the cell below to zip them up and download. Unzip into the repo root on your
computer (both folders are gitignored, so nothing heavy ever lands in git).

In [ ]:
shutil.make_archive("acno_trained", "zip", root_dir=".", base_dir="models")
shutil.make_archive("acno_results", "zip", root_dir=".", base_dir="results")
print("wrote acno_trained.zip and acno_results.zip")

try:
    from google.colab import files
    files.download("acno_trained.zip")
    files.download("acno_results.zip")
except ImportError:
    print("not on Colab, files are in the current folder")

## 9. What we learned

- **Class imbalance was the first real problem**, not model architecture. Both
  classifiers needed class weights before the rare classes (dry skin, whiteheads)
  were predicted at all.
- **Transfer learning made this possible.** A few thousand images is nowhere near
  enough to train a network from scratch, but plenty to fine-tune one.
- **Counting lesions beats guessing a severity label.** The detector's count is
  interpretable, maps onto a real dermatology grading idea, and gives users a number
  they can watch improve.
- **Known limits:** the datasets do not document skin tone coverage, and dermatology
  models are known to do worse on darker skin. Testing on diverse faces before any
  real launch is on our roadmap, and the app always includes a see-a-dermatologist
  path. Guidance, never diagnosis.